# Index Builder

Here we go through the process for creating / updating a vector database from a folder of documents.
* raw files -> chunks -> embeddings -> vector store (Chroma)

We also exercise some care to make this process:
* Idempotent:
  * Deterministic chunk IDs (file_hash:chunk_index)
  * Uses upsert(), so re-running just updates existing records
* Reproducible:
  * All settings are saved in a configuration object
  * All configuration and file names/hashes are saved in a manifest file
    * This manifest (file hashes + config) is saved next to the database

In [ ]:
from pathlib import Path
import hashlib
import json
from pypdf import PdfReader

import chromadb
from sentence_transformers import SentenceTransformer

# Config

Here we create a simple Python dictionary that has all our configuration values.  [For more complex projects, validation, or stronger editor support / type checking, you may find it interesting to explore wrapping this instead as a dataclass or Pydantic model.]

In [ ]:
cfg = {
    'data_dir': "data",              # where your raw docs live
    'persist_dir': "./chromaNB3",       # where Chroma stores its DB
    'collection_name': "docs_v1",

    'embedding_model': "all-MiniLM-L6-v2",
    'chunk_size': 800,              # chars (roughly ~250-330 tokens)
    'chunk_overlap': 200,            # chars of overlap between chunks

    'allowed_exts': (".pdf", ".txt", ".md", ".markdown"),
}

The values can be retrieved simply by accessing keys of this dict:

In [ ]:
cfg['allowed_exts']

In [ ]:
cfg['embedding_model']

# Utility helpers

* `discover_files`: retrieve any file path in a given directory, provided it's a file and it has an allowed file extension
* `compute_file_hash`: uses `hashlib` to generate a unique hash code based on the file contents
* `load_text_document`: simple read function for getting the text contents (here for `.txt` and `.pdf` files)
  * returns dict that include `text` and `metadata` too! (file path, file name, file extension)
* `chunk_text`: does a character-level chunking of a document with overlap

In [ ]:
def discover_files(cfg):
    data_dir = Path(cfg['data_dir'])
    return [
        p for p in data_dir.rglob("*")
        if p.is_file() 
        and p.suffix.lower() in cfg['allowed_exts']
        and not any(i.startswith('.') for i in p.parts)
    ]

In [ ]:
def compute_file_hash(path):

    """SHA256 of file contents for change detection."""
    
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(8192), b""):
            h.update(block)
    return h.hexdigest()

In [ ]:
def load_text_document(path):

    """Simple loader for .txt/.md and .pdf (Swap in other loaders as needed.)"""

    if '.txt' in path.suffix.lower() or '.md' in path.suffix.lower():
        text = path.read_text(encoding="utf-8")
    elif '.pdf' in path.suffix.lower():
        reader = PdfReader(path)
        text = ""
        for page in reader.pages:
            text += page.extract_text() + "\n"  # Add extracted text and a newline
    return {
        "text": text,
        "metadata": {
            "source": str(path.relative_to(path.parents[0])),
            "filename": path.name,
            "ext": path.suffix.lower(),
        },
    }

In [ ]:
def chunk_text(text, chunk_size, chunk_overlap):
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - chunk_overlap
    return chunks

# Manifest for idempotency / reproducibility

Super simple functions for storing the configuration values and file names.
* `load_manifest`: loads a manifest if it exists, or creates a manifest like `{"config": {}, "files": {}}` if it doesn't
* `save_manifest`: creates a parent directory to store the manifest file in if it doesn't exist, and then writes the manifest file

In [ ]:
def load_manifest(manifest_path):
    if not manifest_path.exists():
        return {"config": {}, "files": {}}
    return json.loads(manifest_path.read_text(encoding="utf-8"))

In [ ]:
def save_manifest(manifest_path, manifest):
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

As a toy example of how to use this:

In [ ]:
test_manifest_path = Path('./manifest_dir') / 'test_manifest.json'

In [ ]:
test_manifest_path

In [ ]:
load_manifest(test_manifest_path)

In [ ]:
test_manifest = load_manifest(test_manifest_path)
test_manifest['config'] = cfg
test_manifest['files'] = 'syllabus.pdf'
save_manifest(test_manifest_path, test_manifest)

# Main pipeline

* Look for current files
* Load/create manifest and use that to get file names of previously processed files, if any
* Compute the file hashes
  * this will tell us if previously processed files have changed
  * if nothing's changed, end
* Initialize the embedding model and vector database
* For each changed file:
  * get chunks
  * combine the file-read metadata with more metadata info, including unique chunk ID that has both file hash and chunk index
* Embed the chunk texts
* Add everything into the vector database with `upsert` (inserts a new vector if it does not exist, or updates (replaces) the existing vector if a matching ID or key is found.)
  * The unique chunk ID (file hash and chunk index) is the index for the chunk in the database
* We're done updating the database -- save an updated manifest
  * NOTE: If you change other variables (such as chunk size/overlap, you will need to recreate this database)
  * NOTE: If you remove files from data folder, rather than add, the database will not change.

In [ ]:
def build_index(cfg):

    # 1) Discover files
    files = discover_files(cfg)
    if not files:
        print("No files found. Put some files in the data folder.")
        return

    # 2) Compute hashes and figure out which files changed
    manifest_path = Path(cfg['persist_dir']) / f"{cfg['collection_name']}_manifest.json"
    old_manifest = load_manifest(manifest_path)
    old_files = old_manifest.get("files", {})

    current_files = {}
    changed_files = []

    for path in files:        
        fhash = compute_file_hash(path)
        current_files[str(path)] = fhash
        if old_files.get(str(path)) != fhash:
            changed_files.append(path)
        
    if not changed_files:
        print("No changed files since last run. Nothing to index.")
        # Still update manifest config if needed
        new_manifest = {
            "config": cfg,
            "files": current_files,
        }
        save_manifest(manifest_path, new_manifest)
        return

    print(f"Found {len(changed_files)} changed/new files:")
    for p in changed_files:
        print("  -", p)

    # 3) Init embedding model
    print(f"\nLoading embedding model: {cfg['embedding_model']}")
    model = SentenceTransformer(cfg['embedding_model'])

    # 4) Init Chroma persistent client + collection
    print(f"Connecting to Chroma at {cfg['persist_dir']!r}")
    client = chromadb.PersistentClient(path=cfg['persist_dir'])

    collection = client.get_or_create_collection(cfg['collection_name'])

    # 5) Load, chunk, embed, and upsert for each changed file
    all_ids = []
    all_texts = []
    all_metadatas = []

    for path in changed_files:
        doc = load_text_document(path)
        file_hash = current_files[str(path)]
        chunks = chunk_text(doc["text"], cfg['chunk_size'], cfg['chunk_overlap'])

        print(f"\n{path} -> {len(chunks)} chunks")

        for i, chunk in enumerate(chunks):
            # Deterministic ID: file hash + chunk index
            cid = f"{file_hash}:{i}"
            meta = {
                **doc["metadata"],
                "file_hash": file_hash,
                "chunk_index": i,
                "config_version": cfg['collection_name'],  # simple version tag
            }
            all_ids.append(cid)
            all_texts.append(chunk)
            all_metadatas.append(meta)

    # 6) Embed all chunks in one go (or in batches if huge)
    print(f"\nEmbedding {len(all_texts)} chunks...")
    embeddings = model.encode(all_texts, convert_to_numpy=False)
    embeddings = [vec.tolist() for vec in embeddings]

    # 7) Upsert into the collection (idempotent write)
    print("Upserting into Chroma collection...")
    collection.upsert(
        ids=all_ids,
        documents=all_texts,
        metadatas=all_metadatas,
        embeddings=embeddings,
    )

    # 8) Save updated manifest
    new_manifest = {
        "config": cfg,
        "files": current_files,
    }
    save_manifest(manifest_path, new_manifest)

    print("\nDone. Index is up to date.")

In [ ]:
build_index(cfg)